# Batch OCR with the OpenAI API

This notebook reads images from a folder, previews a sample, asks an OpenAI-compatible vision model to transcribe each image, displays the extracted text, and saves one UTF-8 `.txt` file per image.

Image processing may incur charges. Keep your ELM API key private, and only process images you are permitted to upload.

## 1. Install the required packages

Run this cell once in the notebook environment. The OpenAI SDK version must support the Chat Completions API used by this notebook.

In [ ]:
%pip install httpx==0.27.2 openai==1.55.3 --force-reinstall

## 2. Load your private ELM credentials

Create a `.env` file in the project root (copy `.env.example`) and set both `OPENAI_API_KEY` and the OpenAI-compatible `OPENAI_BASE_URL` supplied by ELM. Do not share or commit `.env`; it is ignored by Git. The notebook loads these values without displaying them. Restart the kernel and run this cell again after changing `.env`.

This notebook uses the Chat Completions API. Confirm that your ELM endpoint supports that API; a base URL alone cannot make an endpoint support a different API.

In [ ]:
from setup import client

print("ELM OpenAI-compatible client is ready.")


## 3. Choose input and output folders

Change `INPUT_DIR` to the folder containing your images. The example points to images included with this repository. Set `RECURSIVE` to `True` to include subfolders. `PREVIEW_LIMIT` caps the number of images shown in the preview cell.

In [ ]:
from pathlib import Path

DOCUMENT = "???" # Change this to the name of your pdf file.

### Set up
INPUT_DIR = Path(f"pdf_pages/{DOCUMENT}")
OUTPUT_DIR = Path(f"pdf_pages/{DOCUMENT}/ORC_output")
MODEL = "gpt-4o-mini" #"gpt-5.6-luna"
RECURSIVE = False
OVERWRITE = False
SUPPORTED_TYPES = {".jpg", ".jpeg", ".png", ".webp", ".gif"}

# Check if the input folder exists
if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"Input folder does not exist: {INPUT_DIR}.")

# Get a list of image files in the input folder
image_paths = sorted(
    path for path in INPUT_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in SUPPORTED_TYPES
)

## Summary of found images
print(f"Found {len(image_paths)} image(s) in {INPUT_DIR}.")
for path in image_paths:
    print(path.relative_to(INPUT_DIR))

## 4. Preview images

The notebook displays images. Increase the limit PREVIEW LIMIT if you want to inspect more before sending them for OCR.

In [ ]:
from IPython.display import display
from PIL import Image

PREVIEW_LIMIT = 5

for image_path in image_paths[:PREVIEW_LIMIT]:
    print(image_path.relative_to(INPUT_DIR))
    with Image.open(image_path) as image:
        display(image.copy())

## 5. Define the OCR request

The model is asked to preserve the language, spelling, and reading order of the visible text rather than translate or summarize it.

In [ ]:
import base64
import mimetypes
from pathlib import Path

from openai import APIConnectionError, OpenAIError

PROMPT_FILE = Path("user_promt.txt")
OCR_INSTRUCTIONS = PROMPT_FILE.read_text(encoding="utf-8").strip()
SUPPORTED_TYPES = {"image/jpeg", "image/png", "image/webp", "image/gif"}



def extract_text_elm(image_path: Path) -> str:
    mime_type, _ = mimetypes.guess_type(image_path.name)
    if mime_type not in SUPPORTED_TYPES:
        raise ValueError(f"Unsupported image type: {image_path.suffix}")

    # Encode the image as a base64 data URL for uploading to the model
    image_data = base64.b64encode(image_path.read_bytes()).decode("ascii")
    image_data_url = f"data:{mime_type};base64,{image_data}"

    # Create a chat completion request to the model with the image and OCRinstructions
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system",
                "content": [
                    {"type": "text", "text": "You are an OCR assistant that extracts text from images."},   
                ],
            },
            {"role": "user",
                "content": [
                    {"type": "text", 
                     "text": OCR_INSTRUCTIONS},

                    {"type": "image_url",
                     "image_url": {"url": image_data_url}},
                    
                ],
            },
        ],
    )

    text = response.choices[0].message.content
    return text.strip()


## 6. Process the batch, display text, and save results

Images are sent sequentially. Results keep the input subfolder structure under `OUTPUT_DIR`. Failed images are reported, and the loop continues so other images can still be processed. Run this cell again to resume; set `OVERWRITE = True` above to reprocess files that already have output.

In [ ]:
from IPython.display import Markdown, display

succeeded = 0
skipped = 0
failed = 0

for index, image_path in enumerate(image_paths, start=1):
    relative_path = image_path.relative_to(INPUT_DIR)
    output_path = (OUTPUT_DIR / relative_path).with_suffix(".txt")

    if output_path.exists() and not OVERWRITE:
        print(f"[{index}/{len(image_paths)}] SKIP {relative_path} (output already exists)")
        skipped += 1
        continue

    try:
        text = extract_text_elm(image_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        output_path.write_text(f"{text}\n", encoding="utf-8")
    except (OpenAIError, OSError, RuntimeError, ValueError) as error:
        if isinstance(error, APIConnectionError):
            cause = error.__cause__
            detail = f" Underlying cause: {cause}" if cause else ""
            message = (
                "APIConnectionError: Could not connect to ELM. Check "
                f"OPENAI_BASE_URL and your network/VPN. {error}{detail}"
            )
        else:
            message = f"{type(error).__name__}: {error}"
        print(f"[{index}/{len(image_paths)}] ERROR {relative_path}: {message}")
        failed += 1
        continue

    print(f"[{index}/{len(image_paths)}] OK {relative_path} -> {output_path}")
    display(Markdown(f"**Extracted text: `{relative_path}`**"))
    print(text)
    
    succeeded += 1

print(
    f"\nFinished: {succeeded} processed, {skipped} skipped, "
    f"{failed} failed. Output folder: {OUTPUT_DIR}"
)